In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, when, sum as _sum

spark = SparkSession.builder.appName("FailedLoginPattern").getOrCreate()

# Sample Data
data = [
    ("U1", "2024-01-01 10:00:00", "fail"),
    ("U1", "2024-01-01 10:05:00", "fail"),
    ("U1", "2024-01-01 10:10:00", "fail"),
    ("U1", "2024-01-01 10:20:00", "success"),
    ("U2", "2024-01-01 11:00:00", "fail"),
    ("U2", "2024-01-01 11:05:00", "fail")
]

columns = ["user_id", "event_time", "status"]

df = spark.createDataFrame(data, columns)

# Step 1: Flag failures
df = df.withColumn("is_fail", when(col("status") == "fail", 1).otherwise(0))

# Step 2: Create group id (reset on success)
window_spec = Window.partitionBy("user_id").orderBy("event_time")

df = df.withColumn(
    "group_id",
    _sum(when(col("status") == "success", 1).otherwise(0)).over(window_spec)
)

# Step 3: Count failures per group
result_df = df.filter(col("is_fail") == 1) \
    .groupBy("user_id", "group_id") \
    .agg(_sum("is_fail").alias("fail_count")) \
    .filter(col("fail_count") >= 3)

result_df.show()